In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated Successfully')

Authenticated Successfully


In [2]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='crafty-airfoil-382608')

In [3]:
create_view_query = """CREATE OR REPLACE VIEW `FICA_Pipeline.vw_Forensic_Alerts_Triage` AS
WITH CTE_First AS (
  SELECT
    Transaction_ID,
    Customer_ID,
    Transaction_Timestamp,
    Channel,
    Amount,
    Merchant_ID,
    Terminal_ID,
    LAG(Channel, 1) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp) AS Previous_Channel,
    LAG(Transaction_Timestamp, 5) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp) AS Fifth_Last_Txn_Time,
    LAG(Transaction_Timestamp, 9) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp) AS Ninth_Last_Txn_Time,
    AVG(Amount) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING) AS Rolling_Avg,
    ROW_NUMBER() OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp) AS Row_Num
  FROM `Fraud_Detection.Fact_Transactions`
),
CTE_Second AS (
  SELECT *,
    LAG(Row_Num) OVER(PARTITION BY Customer_ID, Terminal_ID ORDER BY Transaction_Timestamp) AS Previous_Terminal_Row_Num,
    LAG(Row_Num) OVER(PARTITION BY Customer_ID, Merchant_ID ORDER BY Transaction_Timestamp) AS Previous_Merchant_Row_Num
  FROM CTE_First
),
CTE_Third AS (
  SELECT *,
    TIMESTAMP_DIFF(Transaction_Timestamp, Fifth_Last_Txn_Time, MINUTE) AS Merchant_Time_Window_Minutes,
    TIMESTAMP_DIFF(Transaction_Timestamp, Ninth_Last_Txn_Time, MINUTE) AS Terminal_Time_Window_Minutes,
    SUM(CASE
      WHEN Previous_Terminal_Row_Num IS NULL THEN 1
      WHEN (Row_Num - Previous_Terminal_Row_Num) > 9 THEN 1
      ELSE 0 END) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) AS Terminal_Count,
    SUM(CASE
      WHEN Previous_Merchant_Row_Num IS NULL THEN 1
      WHEN (Row_Num - Previous_Merchant_Row_Num) > 5 THEN 1
      ELSE 0 END) OVER(PARTITION BY Customer_ID ORDER BY Transaction_Timestamp ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS Merchant_Count
  FROM CTE_Second
),
CTE_Fourth AS (
  SELECT *,
    CASE
      WHEN (Merchant_Time_Window_Minutes <= 15 AND Merchant_Count >= 4 AND Amount > (Rolling_Avg * 7)) THEN 'CNPP MERCHANT SWEEP'
      WHEN (Terminal_Time_Window_Minutes <= 15 AND Channel != Previous_Channel AND Terminal_Count >= 6 AND Amount > (Rolling_Avg * 20)) THEN 'STRUCTURED SMASH-AND-GRAB'
      WHEN Amount > (Rolling_Avg * 7) THEN 'SINGLE-SOURCE VOLUME SPIKE'
      ELSE 'NONE'
    END AS Fraud_Type
  FROM CTE_Third
)
SELECT
  Transaction_ID,
  Customer_ID,
  Transaction_Timestamp,
  Channel,
  Amount,
  Merchant_ID,
  Terminal_ID,
  Merchant_Time_Window_Minutes,
  Terminal_Time_Window_Minutes,
  Merchant_Count,
  Terminal_Count,
  Fraud_Type
FROM
  CTE_Fourth
WHERE
  Fraud_Type != 'NONE';"""

client.query(create_view_query).result()
print("View definition updated.")

View definition updated.


In [4]:
data_query = '''SELECT * FROM `FICA_Pipeline.vw_Forensic_Alerts_Triage`'''

fraud_report_df = client.query(data_query).to_dataframe()

if fraud_report_df.empty:
    print("No fraud detected today. System Green.")
else:
    print(f"ALARM: {len(fraud_report_df)} suspicious transactions found!")
    display(fraud_report_df.head())

ALARM: 597 suspicious transactions found!


,Transaction_ID,Customer_ID,Transaction_Timestamp,Channel,Amount,Merchant_ID,Terminal_ID,Merchant_Time_Window_Minutes,Terminal_Time_Window_Minutes,Merchant_Count,Terminal_Count,Fraud_Type
0,96625472,CUST_000050,2026-05-24 00:52:13.896528+00:00,POS_TERMINAL,448.19,MERCH_000268,TERM_000299,<NA>,<NA>,2,2,SINGLE-SOURCE VOLUME SPIKE
1,46401253,CUST_000101,2026-06-20 10:02:00+00:00,POS_TERMINAL,3500.00,MERCH_SWEEP_1,TERM_SWEEP_1,1371,2647,6,10,SINGLE-SOURCE VOLUME SPIKE
2,32889419,CUST_000101,2026-06-20 10:04:00+00:00,ONLINE,3500.00,MERCH_SWEEP_2,TERM_SWEEP_2,606,2386,6,10,SINGLE-SOURCE VOLUME SPIKE
3,96521046,CUST_000101,2026-06-20 10:06:00+00:00,POS_TERMINAL,3500.00,MERCH_SWEEP_3,TERM_SWEEP_3,429,2088,6,10,SINGLE-SOURCE VOLUME SPIKE
4,91616314,CUST_000101,2026-06-20 10:08:00+00:00,ONLINE,3500.00,MERCH_SWEEP_4,TERM_SWEEP_4,225,2048,6,10,SINGLE-SOURCE VOLUME SPIKE


In [5]:
file_name = "Daily_Fraud_Alert_Report.csv"
fraud_report_df.to_csv(file_name, index=False)

print(f"Report generated: {file_name}")

from google.colab import files
files.download(file_name)

Report generated: Daily_Fraud_Alert_Report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>